# 4장 1강: Attention 메커니즘 — 실습문제

## 실습 목표

- Attention과 Self-Attention의 관계를 설명합니다.
- Query·Key·Value의 역할과 배열 크기를 구분합니다.
- 내적 점수를 구하고 √d_k로 나눈 뒤 Softmax와 가중합을 적용합니다.
- Attention 가중치와 최종 출력이 각각 무엇을 뜻하는지 해석합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. 딥러닝 프레임워크는 필요하지 않습니다.
- 기존 **`jena_climate_2009_2016.csv`**를 노트북과 같은 폴더에 두고 그 폴더를 VS Code에서 열어 실행하세요. pandas와 NumPy가 설치된 Python 환경을 커널로 선택합니다.
- 원본은 420,551행, 15열이며, 이번에는 **처음 6행**만 읽습니다. 선택 구간은 10분 간격의 관측입니다.
- 입력 열은 기온 `T (degC)`와 상대습도 `rh (%)`입니다. 시간 `Date Time`은 어떤 관측값인지 표시하는 데 사용합니다.
- 실습은 첫 3개 관측, 과제는 다음 3개 관측을 사용합니다. 한 행은 한 관측 시점, 두 열은 그 시점의 두 특성입니다.
- 입력을 **[기온/10, 상대습도/100]**으로 만듭니다. 계산용 고정 배율 조정이며 원본 데이터는 바꾸지 않습니다. 평균·표준편차를 이용하는 표준화와는 다릅니다.
- 준비 셀부터 위에서 아래로 실행하세요. 계산에는 반올림 전 값을 사용하고 출력에서만 반올림합니다.

## 이번 강의의 범위

제공된 **4장 1강 목차: Attention과 Self-Attention, Query·Key·Value, Scaled Dot-Product Attention**을 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 가중치와 작은 입력 구간은 개념 확인용 예시입니다.

이 자료는 세 관측값 사이의 Attention 계산을 확인합니다. **이미 관측한 구간 전체를 함께 읽는 예시**이며 미래 기온을 예측하는 모델을 학습하지 않습니다. 각 행은 같은 구간의 세 시점을 모두 참고할 수 있습니다. 시간 문자열은 Q·K·V에 들어가지 않으므로 이 계산만으로 관측 간격이나 선후관계까지 학습했다고 해석하지 않습니다.

Q·K·V를 만드는 가중치는 고정해서 제공합니다. 함수 작성, 모델 학습, 미분, Multi-Head Attention, Transformer 전체 구조, 위치 정보 추가, 마스킹의 구현은 요구하지 않습니다. 이번에는 Attention 한 번의 계산과 해석에 집중합니다.

## 실습 구성

| 구분 | 문제 | 할 일 |
|---|---|---|
| 실습 1 | 1-1 | 기상 데이터에서 Q·K·V 만들기 |
| 실습 2 | 2-1 | Scaled Dot-Product Attention 단계별 계산 |
| 실습 2 | 2-2 | 크기 조정 전후의 가중치 비교 |
| 기초 과제 | 3-1 | 다음 3개 관측값에 같은 계산 적용 |

**실습 2개 + 기초 과제 1개**이며, 마지막에는 **실습 마무리 복습 질문 6개**가 있습니다. 실습 2를 전체 계산과 크기 조정의 의미로 나눴습니다. 과제는 같은 식을 다른 구간에 적용하는 필수 수준입니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 예시 |
|---|---|---|
| Attention | Query와 Key를 비교해 반영 비중을 정하고 Value를 가중합하는 방식 | 다른 관측 정보를 어느 비중으로 섞을지 계산 |
| Self-Attention | 같은 입력 묶음에서 Q·K·V를 만드는 Attention | 같은 기상 관측 3행에서 모두 생성 |
| Query, Q | 정보를 모으는 기준이 되는 표현 | 현재 행이 각 Key와 비교될 때 쓰는 숫자 2개 |
| Key, K | Query와 비교되어 점수를 만드는 표현 | 참고할 각 관측 행의 숫자 2개 |
| Value, V | 계산된 비중으로 실제 합쳐질 정보 | 각 행의 기온/10 값 1개 |
| 내적 점수 | Q와 K의 같은 위치를 곱해 더한 값 | q의 두 값과 k의 두 값으로 계산 |
| d_k | Key 한 개를 표현하는 숫자 개수. Q도 같은 차원 | 2 |
| Scaling | 내적 점수를 √d_k로 나누는 것 | √2로 나눔 |
| Softmax | 점수를 양수의 비중으로 바꾸고 합을 1로 맞추는 함수 | Query 한 행마다 적용 |
| Attention 가중치 A | 각 Value를 얼마나 반영할지 나타내는 비중 | 한 행에 가중치 3개 |
| 출력 O | A로 V를 가중합해 얻은 결과 | Query마다 결과 숫자 1개 |

Attention은 상위 개념이고 Self-Attention은 그중 한 방식입니다. 일반적인 Attention에서는 Query의 입력 출처와 Key·Value의 입력 출처가 다를 수도 있습니다. Self-Attention의 ‘Self’는 **같은 입력에서 가져온다**는 뜻이며, 자기 행 하나만 보거나 Q=K=V여야 한다는 뜻이 아닙니다.


### 전체 계산식

$$Q=XW_Q,\quad K=XW_K,\quad V=XW_V$$
$$S=QK^T,\quad Z=S/\sqrt{d_k},\quad A=\operatorname{Softmax}(Z),\quad O=AV$$

Softmax는 각 행에서 계산합니다.

$$A_{ij}=\frac{\exp(Z_{ij})}{\sum_j\exp(Z_{ij})}$$

`i`는 정보를 모으는 Query 행, `j`는 참고할 Key·Value 행입니다. 분모는 **같은 i행의 모든 j**에 대해 더합니다. 각 Query가 참고하는 값들의 비중 합을 1로 만드는 것입니다.

표기 `@`는 행렬곱, `.T`는 행과 열을 바꾸는 전치, `np.sqrt()`는 제곱근입니다. 이번 내용은 앞서 배운 가중합과 행렬곱을 사용합니다.


### 왜 Attention을 배우나요?

여러 관측값을 하나로 요약할 때 모든 값을 똑같이 반영하는 평균을 사용할 수 있습니다. Attention은 **어떤 관측을 기준으로 정보를 모으는지에 따라, 참고할 값들의 비중을 계산하는 방식**입니다. 같은 세 관측을 참고하더라도 기준이 되는 Query가 달라지면 비중과 요약 결과가 달라질 수 있습니다.

예를 들어 세 값이 10, 20, 30이고 반영 비중이 0.2, 0.3, 0.5라면 `0.2×10 + 0.3×20 + 0.5×30 = 23`입니다. 이처럼 비중을 곱해 더하는 것을 **가중합**이라고 합니다. 여기서는 비중이 모두 0 이상이고 합이 1이므로 가중평균으로도 읽을 수 있습니다. Attention은 이 비중을 Q와 K의 비교로 계산합니다. 위의 0.2·0.3·0.5는 계산 방법을 설명하기 위한 숫자이며 실제 실습의 정답은 아닙니다.


### 벡터·행렬·기호부터 읽기

벡터는 숫자를 순서대로 모은 것이고, 행렬은 그런 숫자를 행과 열로 배열한 것입니다. 이번에는 한 시점의 `[기온/10, 상대습도/100]`이 숫자 두 개를 가진 벡터이며, 세 시점을 세 행으로 모으면 입력 행렬 X가 됩니다.

| 표현 | 쉽게 읽기 | 이번 값 또는 역할 |
|---|---|---|
| `(3, 2)` | 3행, 2열 | 관측 3개, 관측마다 특성 2개 |
| Q, K, V | 여러 관측의 표현을 모은 행렬 | 서로 다른 세 가지 계산 역할 |
| q1, k1, v1 | 각 행렬의 첫 번째 행 | 00:10 관측에서 만든 표현 |
| $W_Q,W_K,W_V$ | Q·K·V를 만드는 가중치 행렬 | 입력의 각 성분을 어떻게 결합할지 정함 |
| $K^T$ | K의 전치 | 행을 열로 바꾸어 모든 Q·K 쌍을 비교할 준비 |
| $d_k$ | Key 한 행의 성분 수 | 이 예시는 2이며, 관측 수 3과 다름 |
| $\sqrt{d_k}$ | d_k의 제곱근 | 제곱하면 d_k가 되는 양수; √2 ≈ 1.414214 |
| S, Z, A, O | 점수, 조정한 점수, 비중, 출력 | 서로 다른 계산 단계의 결과 |
| $A_{ij}$ | A의 i번째 행, j번째 열 | i번째 Query가 j번째 Value에 주는 비중 |
| $\sum_j$ | j를 바꾸며 모두 더하기 | 같은 Query 행의 세 값을 합함 |
| $\exp(z)$ | 지수함수 e의 z제곱 | Softmax에서 점수를 양수로 바꾸는 계산 |

`exp`의 e는 약 2.718인 수입니다. `exp(0)=1`이고 입력이 커질수록 결과도 커집니다. 이 수업에서는 함수값을 손으로 구하지 않고 `np.exp()`나 제공 Softmax 함수로 확인하면 됩니다.

수식에서는 첫 행을 q1이라고 부르지만, Python의 첫 행 번호는 0이라 `Q[0]`으로 선택합니다. `A[0, 2]`는 첫 번째 Query가 세 번째 Value에 주는 비중입니다. 아래첨자 i·j는 제곱을 뜻하지 않고 행·열 번호를 구분하는 표시입니다.


### 주제별 학습 위치

| 강의 주제 | 확인할 문제 | 설명할 수 있어야 하는 내용 |
|---|---|---|
| Attention과 Self-Attention | 실습 1-1 Q1, 실습 2-1 | 비중으로 정보 결합, 같은 입력에서 Q·K·V 생성 |
| Query·Key·Value | 실습 1-1, 과제 3-1 | 각 표현의 역할, 크기, 점수와 결합에 쓰이는 위치 |
| Scaled Dot-Product Attention | 실습 2-1·2-2, 과제 3-1 | 내적 → 크기 조정 → Softmax → Value 가중합 |

문제는 수식을 암기하여 새 알고리즘을 구현하기보다, 제공된 계산을 실행하고 각 단계의 의미를 설명하는 기초 수준입니다.


## 실습 준비 — 두 코드 셀을 그대로 실행하세요

첫 셀은 데이터와 가중치를 준비합니다. 한 입력 행을 `[a, b]=[기온/10, 상대습도/100]`이라고 하면 다음과 같이 변환됩니다.

| 변환 | 가중치 행렬 | 한 행의 결과 |
|---|---|---|
| Query | WQ = [[1, 0], [0, 1]] | [a, b] |
| Key | WK = [[1, 0], [0, 0.5]] | [a, 0.5b] |
| Value | WV = [[1], [0]] | [a] |

가중치는 계산을 따라가기 쉽게 정한 값이며 학습한 결과가 아닙니다. 일반적인 모델에서는 이런 변환 가중치를 학습합니다. **WQ·WK·WV와 Attention 가중치 A는 다릅니다.** 전자는 Q·K·V를 만드는 파라미터이고, 후자는 현재 입력의 점수에서 계산되는 반영 비중입니다.

두 번째 셀은 **행별 Softmax 제공 함수**입니다. 함수를 다시 작성하는 것은 문제가 아닙니다. `axis=1`은 각 행의 열 방향으로 계산한다는 뜻입니다. `keepdims=True`는 결과를 세로 한 열 모양으로 유지하여 원래 행렬의 각 행에 맞춰 계산하게 합니다.

함수는 지수 계산 전에 각 행의 최댓값을 뺍니다. 같은 행의 모든 점수에서 같은 수를 빼도 Softmax 비율은 같으며, 큰 수의 지수 계산을 피하는 데 도움이 됩니다. 증명은 요구하지 않습니다.

### 코드와 행렬 크기 읽기

- `=`는 오른쪽 계산 결과를 왼쪽 이름에 저장한다는 뜻입니다. `Q = X @ WQ`는 행렬곱 결과를 Q에 저장합니다.
- `@`는 행렬곱입니다. 왼쪽 행과 오른쪽 열에서 대응하는 숫자끼리 곱하고 더해 출력 한 칸을 만듭니다. 배열의 같은 위치끼리 곱하는 `*`와 다릅니다.
- `(3, 2) @ (2, 2)`에서 가운데 2가 같으므로 곱할 수 있고 결과는 바깥 크기 `(3, 2)`입니다. `(3, 2) @ (2, 1)`의 결과는 `(3, 1)`입니다.
- `K.shape[1]`은 K의 열 개수입니다. `shape[0]`은 행 개수이므로 여기서는 d_k를 구할 때 사용하지 않습니다.
- `.T`는 전치입니다. 예를 들어 행이 `[1, 2]`, `[3, 4]`인 행렬을 전치하면 행이 `[1, 3]`, `[2, 4]`가 됩니다.
- `axis=1`로 합을 구하면 각 행의 값을 가로 방향으로 더합니다. 이번에는 Query별 비중 합을 확인합니다.
- `np.round(값, 6)`은 소수 여섯 자리로 반올림한 표시용 값을 만듭니다. 중간 계산에는 반올림 전 숫자를 사용합니다.

준비 셀을 실행한 뒤 실습 1부터 차례로 실행하세요. 뒤 문제의 `Q`, `scores`, `weights` 등은 앞 문제에서 만든 결과를 사용합니다.


### 처음 코드를 읽는 학생을 위한 안내

| 표현 | 읽는 방법 | 이번 실습의 예 |
|---|---|---|
| `# 설명` | 사람이 읽는 주석이며 계산하지 않습니다. | 각 줄의 목적을 알려 줍니다. |
| `이름 = 계산` | 오른쪽 결과를 왼쪽 이름에 저장합니다. | `scores = Q @ K.T`는 점수 행렬을 저장합니다. |
| `np.array([...])` | 여러 숫자를 계산 가능한 배열로 만듭니다. | 가중치 행렬을 준비합니다. |
| `[[...], [...]]` | 안쪽 목록 하나가 한 행인 2차원 배열입니다. | `[[1], [0]]`은 2행·1열입니다. |
| `.shape` | 배열의 행·열 크기를 확인합니다. | `(3, 2)`는 3행·2열입니다. |
| `[0]`, `[0, 0]` | 첫 행 또는 첫 행·첫 열을 선택합니다. | Python의 위치 번호는 0부터 시작합니다. |
| `[:, 0]` | 모든 행에서 첫 열만 선택합니다. | V에 있는 세 숫자를 한 줄의 배열로 꺼냅니다. |
| `def 함수(...):` | 나중에 사용할 처리 방법을 정의합니다. | `softmax_rows`의 계산 순서를 준비합니다. |
| `return 결과` | 함수의 계산 결과를 호출한 곳에 돌려줍니다. | `weights = softmax_rows(...)`로 결과를 받습니다. |
| `print(...)` | 값을 화면에 표시합니다. | 표와 행렬의 결과를 확인합니다. |

준비 셀은 그대로 실행하고, 실습 1부터 순서대로 풀어 보세요. 실습 2는 앞에서 만든 Q·K·V를 사용합니다. `NameError`로 변수 이름을 찾지 못한다면, 필요한 값을 만드는 앞 셀을 실행했는지 먼저 확인하세요.

**배열의 값과 배열의 크기를 구분하세요.** `Q[0]`은 숫자들이 들어 있는 첫 행이고 `Q.shape`는 행·열 개수입니다. `K.shape[1]`의 1은 두 번째 항목인 **열 개수**를 고르는 위치 번호이며, Key의 첫 값이나 두 번째 관측값을 고르는 코드가 아닙니다.


In [12]:
# 표 형태의 데이터를 다루는 pandas를 pd라는 이름으로 불러옵니다.
import pandas as pd
# 숫자 배열 계산을 위해 NumPy를 np라는 이름으로 불러옵니다.
import numpy as np

# 노트북과 같은 폴더의 CSV에서 앞 여섯 행을 읽습니다.
df = pd.read_csv("jena_climate_2009_2016.csv", nrows=6)
# 관측 시각을 날짜·시간 자료로 바꿉니다. 일.월.년 순서입니다.
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
# 기온, 상대습도 순으로 두 열을 선택해 6행·2열 배열로 만듭니다.
raw = df[["T (degC)", "rh (%)"]].to_numpy(dtype=float)
# 각 행의 첫 열은 10으로, 둘째 열은 100으로 나눕니다.
X_all = raw / np.array([10.0, 100.0])
# 위치 0~2의 세 관측을 실습 입력으로 선택합니다.
X = X_all[:3]
# 위치 3~5의 세 관측을 과제 입력으로 선택합니다.
X_home = X_all[3:6]

# 두 입력 성분을 그대로 Q의 두 성분으로 만드는 예시 가중치입니다.
WQ = np.array([[1.0, 0.0], [0.0, 1.0]])
# 둘째 입력 성분을 절반으로 바꾸는 K용 가중치입니다.
WK = np.array([[1.0, 0.0], [0.0, 0.5]])
# 첫 입력 성분만 V에 남기는 2행·1열 가중치입니다.
WV = np.array([[1.0], [0.0]])

# 읽은 표의 행·열 개수를 확인합니다.
print("읽은 데이터 크기:", df.shape)
# 시간·기온·습도만 화면에 보여 줍니다.
print(df[["Date Time", "T (degC)", "rh (%)"]].to_string(index=False))
# 시각 차이를 초로 바꾸고 60으로 나누어 분 단위로 확인합니다.
print("\n 선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
# 첫 sum은 열별 빈칸 수, 둘째 sum은 두 열의 합계를 구합니다.
print("\n선택 두 열의 결측값 수:", df[["T (degC)", "rh (%)"]].isna().sum().sum())
# 실습 배열의 각 행이 한 시점인지 확인합니다.
print("\n실습 X:\n", X)
# 과제 배열은 준비 단계에서 이미 배율 조정이 끝났습니다.
print("\n과제 X_home:\n", X_home)
# 세 가중치 행렬의 크기를 확인합니다.
print("\n가중치 크기:", WQ.shape, WK.shape, WV.shape)


읽은 데이터 크기: (6, 15)
          Date Time  T (degC)  rh (%)
01.01.2009 00:10:00     -8.02    93.3
01.01.2009 00:20:00     -8.41    93.4
01.01.2009 00:30:00     -8.51    93.9
01.01.2009 00:40:00     -8.31    94.2
01.01.2009 00:50:00     -8.27    94.1
01.01.2009 01:00:00     -8.05    94.4

 선택 구간 간격(분): [10.]

선택 두 열의 결측값 수: 0

실습 X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]]

과제 X_home:
 [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]]

가중치 크기: (2, 2) (2, 2) (2, 1)


In [2]:
# 점수 행렬의 각 행에 Softmax를 적용하는 제공 함수입니다.
def softmax_rows(scores):
    # 행마다 최댓값을 찾아 그 행의 모든 점수에서 뺍니다. 비중은 바뀌지 않습니다.
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    # 각 숫자에 지수 함수를 적용합니다. exp(0)=1입니다.
    exp_scores = np.exp(shifted)
    # 각 행의 지수값 합으로 나누어 비중 합을 1로 맞춥니다.
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


print("행별 Softmax 함수 준비 완료")


행별 Softmax 함수 준비 완료


---

## 실습 1. Attention과 Query·Key·Value


### 문제 1-1. 같은 입력에서 Q·K·V 만들기

#### 문제 설명

입력 X는 세 관측 시점의 기온과 상대습도를 담고 있습니다. 준비된 세 가중치 행렬을 사용해 Q·K·V를 각각 만드세요.

#### 요구사항

1. `Q=X@WQ`, `K=X@WK`, `V=X@WV`를 계산하세요.
2. X·Q·K·V의 값과 크기를 출력하세요.
3. 첫 번째 관측 행으로 만든 q1·k1·v1을 직접 계산하세요. 대문자는 여러 행을 모은 행렬, 소문자는 그중 한 행을 뜻합니다.

#### 작성 도움말

`print(Q)`는 값, `print(Q.shape)`는 행·열 크기를 보여줍니다. 첫 행은 `Q[0]`입니다. 첫 입력은 `[-0.802, 0.933]`입니다.

#### 해석 질문

**Q1.** Attention과 Self-Attention은 어떤 관계인가요? 이번 계산이 Self-Attention인 이유는 무엇인가요?

**Q2.** Q·K·V의 크기와 첫 행의 값은 각각 무엇인가요? Q와 K는 두 값인데 V는 한 값이어도 되나요?

**Q3.** Q·K·V는 각각 어떤 역할을 하나요? 이번 가중치 행렬은 학습한 결과인가요? WQ·WK·WV와 이후에 구할 A는 어떻게 다른가요?

#### 제출 결과

행렬 값·크기, 첫 행 변환 계산, Q1~Q3 답변.


In [13]:
# 제공된 가중치로 Q, K, V를 만드세요.
# Q = ...
# K = ...
# V = ...

# 각 값과 shape, 첫 행을 출력하세요.

# 1. `Q=X@WQ`, `K=X@WK`, `V=X@WV`를 계산하세요.
Q = X @ WQ
K = X @ WK
V = X @ WV

# 2. X·Q·K·V의 값과 크기를 출력하세요.
print("X:\n", X, "\nX shape:", X.shape)
print("\nQ:\n", Q, "\nQ shape:", Q.shape)
print("\nK:\n", K, "\nK shape:", K.shape)
print("\nV:\n", V, "\nV shape:", V.shape)

# 3. 첫 번째 관측 행으로 만든 q1·k1·v1을 직접 계산하세요. 대문자는 여러 행을 모은 행렬, 소문자는 그중 한 행을 뜻합니다.
x1 = X[0]
q1 = x1 @ WQ
k1 = x1 @ WK
v1 = x1 @ WV
print("\nx1:", x1)
print("q1:", q1, "/ Q[0]:", Q[0])
print("k1:", k1, "/ K[0]:", K[0])
print("v1:", v1, "/ V[0]:", V[0])


X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] 
X shape: (3, 2)

Q:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] 
Q shape: (3, 2)

K:
 [[-0.802   0.4665]
 [-0.841   0.467 ]
 [-0.851   0.4695]] 
K shape: (3, 2)

V:
 [[-0.802]
 [-0.841]
 [-0.851]] 
V shape: (3, 1)

x1: [-0.802  0.933]
q1: [-0.802  0.933] / Q[0]: [-0.802  0.933]
k1: [-0.802   0.4665] / K[0]: [-0.802   0.4665]
v1: [-0.802] / V[0]: [-0.802]


#### 답변 작성란

**수식 및 계산 과정**

- 작성:  
  - **Q = X @ WQ** (WQ = [[1, 0], [0, 1]] → 입력을 그대로 둠)  
    -> Q = [[-0.802, 0.933], [-0.841, 0.934], [-0.851, 0.939]], 크기 (3, 2)  
  - **K = X @ WK** (WK = [[1, 0], [0, 0.5]] → 습도만 절반으로)  
    -> K = [[-0.802, 0.4665], [-0.841, 0.467], [-0.851, 0.4695]], 크기 (3, 2)  
  - **V = X @ WV** (WV = [[1], [0]] → 기온만 남김)  
    -> V = [[-0.802], [-0.841], [-0.851]], 크기 (3, 1)  
  - **첫 행 계산 (x1 = [-0.802, 0.933])**  
    -> q1 = [-0.802 × 1 + 0.933 × 0, -0.802 × 0 + 0.933 × 1] = [-0.802, 0.933]  
    -> k1 = [-0.802 × 1 + 0.933 × 0, -0.802 × 0 + 0.933 × 0.5] = [-0.802, 0.4665]  
    -> v1 = [-0.802 × 1 + 0.933 × 0] = [-0.802]  

**Q1. Attention과 Self-Attention은 어떤 관계인가요? 이번 계산이 Self-Attention인 이유는 무엇인가요?**

- 답변:  
  - **Self-Attention은 Attention의 한 종류임**  
    -> Attention은 Q로 K와의 비교로 비중을 구하고 V를 결합함  
    -> self-attention은 QKV의 출처가 같은 입력 묶음인 경우를 말한다.  

**Q2. Q·K·V의 크기와 첫 행의 값은 각각 무엇인가요? Q와 K는 두 값인데 V는 한 값이어도 되나요?**

- 답변:  
  - **크기** : Q (3, 2), K (3, 2), V (3, 1)  
  - **첫 행** : q1 = [-0.802, 0.933], k1 = [-0.802, 0.4665], v1 = [-0.802]  
  - **V는 한 값이어도 됨**  
    -> Q와 K는 서로 곱해서(내적) 관련도 점수를 내야 하므로 **길이가 같아야 함**  
    -> V는 점수 계산에 쓰이지 않고 **가져올 정보**만 담으므로 V의 성분 수는 다를 수 있다.  

**Q3. Q·K·V는 각각 어떤 역할을 하나요? 이번 가중치 행렬은 학습한 결과인가요? WQ·WK·WV와 이후에 구할 A는 어떻게 다른가요?**

- 답변:  
  - **Q (Query)** : 정보를 모으는 기준  
  - **K (Key)** : 그 기준과 비교되는 표현  
  - **V (Value)** : 실제 가중합에 들어갈 정보  
  - **학습한 결과가 아님**  
    -> 계산 과정을 보여주려고 문제에서 정해 준 예시 가중치임 (그대로 두기, 습도 절반, 기온만 남기기)  
  - **WQ·WK·WV와 A의 차이**  
    -> WQ·WK·WV : 입력을 Q·K·V로 바꾸는 **가중치(파라미터)** 로, 실제 모델에서는 학습으로 정해지고 학습이 끝나면 어떤 입력이 와도 같은 값을 씀  
    -> A : Q와 K로 계산한 점수에 Softmax를 적용한 **Attention 비중** 으로, 학습하는 값이 아니라 **입력이 바뀔 때마다 새로 계산**되며 각 행의 합이 1임

---

## 실습 2. Scaled Dot-Product Attention


### 문제 2-1. 내적부터 Value 가중합까지 계산하기

#### 문제 설명

실습 1의 Q·K·V를 이용해 Attention의 네 단계를 계산하세요.

$$S=QK^T\quad\rightarrow\quad Z=S/\sqrt{d_k}\quad\rightarrow\quad A=\operatorname{Softmax}(Z)\quad\rightarrow\quad O=AV$$

이번에는 d_k=2입니다. A의 i행 j열은 **i번째 Query가 j번째 Value에 주는 비중**입니다. K는 그 비중의 점수를 정하는 데 사용합니다.

#### 요구사항

1. `scores=Q@K.T`로 모든 Query·Key 쌍의 내적을 구하세요.
2. `d_k=K.shape[1]`로 Key의 성분 수를 확인하고 `scaled_scores=scores/np.sqrt(d_k)`를 계산하세요.
3. 제공 함수 `softmax_rows(scaled_scores)`로 `weights`를 구하세요.
4. `output=weights@V`로 Value를 가중합하세요.
5. 네 단계의 값과 크기, 가중치의 행별 합을 출력하세요.
6. 첫 Query 행의 점수 3개와 최종 출력 하나를 직접 대입해 계산하세요. Softmax 숫자는 제공 함수로 확인해도 됩니다.

#### 작성 도움말

`weights.sum(axis=1)`로 각 행의 가중치 합을 구합니다. 첫 Query의 출력은 `weights[0,0]*V[0,0]+weights[0,1]*V[1,0]+weights[0,2]*V[2,0]`입니다. 코드의 첫 행·첫 열 번호는 0입니다.

#### 해석 질문

**Q1.** 첫 Query와 세 Key의 내적 점수는 각각 얼마인가요? 첫 내적의 곱셈·덧셈도 쓰세요.

**Q2.** 첫 Query의 Attention 가중치 3개와 그 합은 얼마인가요? 세 행렬 S·Z·A의 크기와 최종 O의 크기는 무엇인가요?

**Q3.** 첫 Query의 가중합 결과는 얼마인가요? Attention 가중치는 양수인데 출력이 음수일 수 있는 이유는 무엇인가요?

**Q4.** 첫 Query가 가장 큰 비중을 주는 관측은 무엇인가요? 그 값이 가장 크다는 사실만으로 해당 시점이 미래 기온의 원인이라고 결론 내릴 수 있나요?

#### 제출 결과

단계별 행렬·크기, 행별 가중치 합, 첫 행 대입 계산, Q1~Q4 답변.


In [15]:
# d_k는 Key 한 행의 성분 수입니다.
# d_k = ...
# scores = ...
# scaled_scores = ...
# weights = ...
# output = ...

# 각 값과 shape, weights.sum(axis=1)을 출력하세요.
# 첫 출력에 대한 Value별 기여도도 확인하세요.

# 1. `scores=Q@K.T`로 모든 Query·Key 쌍의 내적을 구하세요.
scores = Q @ K.T

# 2. `d_k=K.shape[1]`로 Key의 성분 수를 확인하고 `scaled_scores=scores/np.sqrt(d_k)`를 계산하세요.
d_k = K.shape[1]
scaled_scores = scores / np.sqrt(d_k)

# 3. 제공 함수 `softmax_rows(scaled_scores)`로 `weights`를 구하세요.
weights = softmax_rows(scaled_scores)

# 4. `output=weights@V`로 Value를 가중합하세요.
output = weights @ V

# 5. 네 단계의 값과 크기, 가중치의 행별 합을 출력하세요.
print("d_k:", d_k)
print("\n[S] scores:\n", scores.round(6), "\nshape:", scores.shape)
print("\n[Z] scaled_scores:\n", scaled_scores.round(6), "\nshape:", scaled_scores.shape)
print("\n[A] weights:\n", weights.round(6), "\nshape:", weights.shape)
print("\n[O] output:\n", output.round(6), "\nshape:", output.shape)
print("\n가중치 행별 합:", weights.sum(axis=1).round(6))

# 6. 첫 Query 행의 점수 3개와 최종 출력 하나를 직접 대입해 계산하세요. Softmax 숫자는 제공 함수로 확인해도 됩니다.
print("\n[첫 Query 점수]")
for j in range(3):
    s = Q[0, 0] * K[j, 0] + Q[0, 1] * K[j, 1]
    print(f"q1·k{j+1} = ({Q[0,0]:.4f})×({K[j,0]:.4f}) + ({Q[0,1]:.4f})×({K[j,1]:.4f}) = {s:.6f}")

print("\n[첫 출력에 대한 Value별 기여도]")
contrib = weights[0] * V[:, 0]
for j in range(3):
    print(f"A[0,{j}] × V[{j},0] = {weights[0, j]:.6f} × ({V[j, 0]:.4f}) = {contrib[j]:.6f}")
print(f"첫 출력 o1 = {contrib.sum():.6f} / output[0,0] = {output[0, 0]:.6f}")


d_k: 2

[S] scores:
 [[1.078448 1.110193 1.120546]
 [1.110193 1.143459 1.154204]
 [1.120546 1.154204 1.165062]] 
shape: (3, 3)

[Z] scaled_scores:
 [[0.762578 0.785025 0.792345]
 [0.785025 0.808548 0.816145]
 [0.792345 0.816145 0.823823]] 
shape: (3, 3)

[A] weights:
 [[0.327556 0.334991 0.337453]
 [0.327288 0.335078 0.337634]
 [0.327218 0.3351   0.337682]] 
shape: (3, 3)

[O] output:
 [[-0.8316  ]
 [-0.831612]
 [-0.831615]] 
shape: (3, 1)

가중치 행별 합: [1. 1. 1.]

[첫 Query 점수]
q1·k1 = (-0.8020)×(-0.8020) + (0.9330)×(0.4665) = 1.078448
q1·k2 = (-0.8020)×(-0.8410) + (0.9330)×(0.4670) = 1.110193
q1·k3 = (-0.8020)×(-0.8510) + (0.9330)×(0.4695) = 1.120545

[첫 출력에 대한 Value별 기여도]
A[0,0] × V[0,0] = 0.327556 × (-0.8020) = -0.262700
A[0,1] × V[1,0] = 0.334991 × (-0.8410) = -0.281728
A[0,2] × V[2,0] = 0.337453 × (-0.8510) = -0.287172
첫 출력 o1 = -0.831600 / output[0,0] = -0.831600


#### 답변 작성란

**수식 및 계산 과정**

- 작성:  
  - **1단계 S = Q @ Kᵀ** (크기 (3, 3))  
    -> 첫 행 : [1.078449, 1.110193, 1.120545]  
  - **2단계 Z = S / √2** (d_k = 2, √2 ≈ 1.414214)  
    -> 첫 행 : [1.078449 / 1.414214, 1.110193 / 1.414214, 1.120545 / 1.414214] = [0.762578, 0.785025, 0.792345]  
  - **3단계 A = Softmax(Z)** (각 값의 exp를 행 합계로 나눔)  
    -> exp 값 : [2.143796, 2.192462, 2.208570], 합 6.544828  
    -> 첫 행 : [2.143796 / 6.544828, 2.192462 / 6.544828, 2.208570 / 6.544828] = [0.327556, 0.334991, 0.337453]  
  - **4단계 O = A @ V** (크기 (3, 1))  
    -> o1 = 0.327556 × (-0.802) + 0.334991 × (-0.841) + 0.337453 × (-0.851)  
    -> = -0.262700 + (-0.281728) + (-0.287172) = -0.831600  

**Q1. 첫 Query와 세 Key의 내적 점수는 각각 얼마인가요? 첫 내적의 곱셈·덧셈도 쓰세요.**

- 답변:  
  - **q1·k1 = 1.078449, q1·k2 = 1.110193, q1·k3 = 1.120545**  
  - **첫 내적** (q1 = [-0.802, 0.933], k1 = [-0.802, 0.4665])  
    -> q1·k1 = (-0.802) × (-0.802) + 0.933 × 0.4665 = 0.643204 + 0.435245 = 1.078449  
  - 나머지 두 개도 같은 방식으로 계산함  
    -> q1·k2 = (-0.802) × (-0.841) + 0.933 × 0.467 = 0.674482 + 0.435711 = 1.110193  
    -> q1·k3 = (-0.802) × (-0.851) + 0.933 × 0.4695 = 0.682502 + 0.438044 = 1.120545 (반올림 차이)  

**Q2. 첫 Query의 Attention 가중치 3개와 그 합은 얼마인가요? 세 행렬 S·Z·A의 크기와 최종 O의 크기는 무엇인가요?**

- 답변:  
  - **첫 Query 가중치 : [0.327556, 0.334991, 0.337453], 합 1.0**  
    -> Softmax는 각 행을 합이 1인 비중으로 바꾸므로, 세 행 모두 합이 1임  
  - **크기** : S (3, 3), Z (3, 3), A (3, 3), O (3, 1)  
  -> softmax는 행 별 합을 맞추기 떄문에 전체 9개의 가중합은 3


**Q3. 첫 Query의 가중합 결과는 얼마인가요? Attention 가중치는 양수인데 출력이 음수일 수 있는 이유는 무엇인가요?**

- 답변:  
  - **첫 Query의 가중합 결과 : -0.831600**  
    -> 0.327556 × (-0.802) + 0.334991 × (-0.841) + 0.337453 × (-0.851) = -0.831600  
  - **출력이 음수인 이유** : 양수 가중치로 음수 value들을 곱하고 더했기 떄문에 음수일 수 있다.  
  - 가중치가 양수라는 조건이 출력까지 양수로 만들지 않음  

**Q4. 첫 Query가 가장 큰 비중을 주는 관측은 무엇인가요? 그 값이 가장 크다는 사실만으로 해당 시점이 미래 기온의 원인이라고 결론 내릴 수 있나요?**

- 답변:  
  - **가장 큰 비중 : 세 번째 관측 (00:30, -8.51도, 습도 93.9%), 비중 0.337453**  
    -> 다른 비중과의 차이가 작고, 사람이 정한 가중치에서 계산한 상대적 비중을 의미

### 문제 2-2. 왜 내적 점수를 √d_k로 나눌까요?

#### 문제 설명

문제 2-1의 `scores`를 그대로 사용해, 크기 조정 없이 Softmax를 적용한 경우와 비교하세요. 새로운 데이터나 가중치는 사용하지 않습니다.

- 크기 조정 있음: `softmax_rows(scores / sqrt(d_k))`
- 크기 조정 없음: `softmax_rows(scores)`

두 경우 모두 Softmax를 적용합니다. 이번에 비교할 것은 **내적 점수를 √d_k로 나누는 단계의 유무**입니다.

#### 요구사항

1. `weights_no_scale=softmax_rows(scores)`를 계산하세요.
2. 첫 Query의 조정 전후 가중치를 표로 비교하세요.
3. 두 경우의 첫 행 최댓값과 행 합을 출력하세요.
4. 크기 조정의 목적과 이번 숫자의 차이를 설명하세요.

#### 해석 질문

**Q1.** 이번에는 왜 √3이나 √1이 아니라 √2로 나누나요? 첫 행 가중치의 최댓값은 조정 전후에 각각 얼마인가요?

**Q2.** Scaled Dot-Product Attention에서 크기 조정은 왜 하나요? 이번 예시에서 차이가 아주 크다고 말할 수 있나요?

**Q3.** √d_k로 나누는 것만으로 가중치 합이 1이 되나요? 한 Query의 점수가 모두 같다면 Softmax 가중치는 어떻게 되나요?

#### 제출 결과

가중치 비교 표, 첫 행 최댓값·합, Q1~Q3 답변.


In [16]:
# 크기 조정 없이 Softmax를 적용하세요.
# weights_no_scale = ...

# 첫 Query 행의 두 가중치 배열을 비교하세요.
# 각각의 최댓값과 합을 출력하세요.

# 1. `weights_no_scale=softmax_rows(scores)`를 계산하세요.
weights_no_scale = softmax_rows(scores)

# 2. 첫 Query의 조정 전후 가중치를 표로 비교하세요.
compare = pd.DataFrame({
    "Key": ["k1", "k2", "k3"],
    "크기 조정 있음": weights[0],
    "크기 조정 없음": weights_no_scale[0],
})
compare["차이(없음-있음)"] = compare["크기 조정 없음"] - compare["크기 조정 있음"]
print(compare.round(6).to_string(index=False))

# 3. 두 경우의 첫 행 최댓값과 행 합을 출력하세요.
print("\n[크기 조정 있음] 첫 행 최댓값:", round(weights[0].max(), 6), "/ 행 합:", round(weights[0].sum(), 6))
print("[크기 조정 없음] 첫 행 최댓값:", round(weights_no_scale[0].max(), 6), "/ 행 합:", round(weights_no_scale[0].sum(), 6))

# 4. 크기 조정의 목적과 이번 숫자의 차이를 설명하세요.
print("\n[첫 행 최댓값 - 최솟값]")
print("크기 조정 있음:", round(weights[0].max() - weights[0].min(), 6))
print("크기 조정 없음:", round(weights_no_scale[0].max() - weights_no_scale[0].min(), 6))

Key  크기 조정 있음  크기 조정 없음  차이(없음-있음)
 k1  0.327556  0.325177  -0.002379
 k2  0.334991  0.335665   0.000674
 k3  0.337453  0.339158   0.001705

[크기 조정 있음] 첫 행 최댓값: 0.337453 / 행 합: 1.0
[크기 조정 없음] 첫 행 최댓값: 0.339158 / 행 합: 1.0

[첫 행 최댓값 - 최솟값]
크기 조정 있음: 0.009897
크기 조정 없음: 0.013981


#### 답변 작성란

**수식 및 계산 과정**

- 작성:  
  - **크기 조정 있음** : Softmax(S / √2)  
    -> 첫 행 Z = [0.762578, 0.785025, 0.792345]  
    -> 가중치 = [0.327556, 0.334991, 0.337453]  
  - **크기 조정 없음** : Softmax(S)  
    -> 첫 행 S = [1.078448, 1.110193, 1.120546]  
    -> exp 값 = [2.940114, 3.034944, 3.066527], 합 9.041585  
    -> 가중치 = [2.940114 / 9.041585, 3.034944 / 9.041585, 3.066527 / 9.041585] = [0.325177, 0.335665, 0.339158]  

| Key | 크기 조정 있음 | 크기 조정 없음 | 차이(없음-있음) |
|---|---|---|---|
| k1 | 0.327556 | 0.325177 | -0.002379 |
| k2 | 0.334991 | 0.335665 | 0.000674 |
| k3 | 0.337453 | 0.339158 | 0.001705 |

**Q1. 이번에는 왜 √3이나 √1이 아니라 √2로 나누나요? 첫 행 가중치의 최댓값은 조정 전후에 각각 얼마인가요?**

- 답변:  
  - **√2로 나누는 이유** : d_k는 Key 한 행의 성분 수, 예제에서는 K의 크기가 (3, 2)라 d_k = 2이기 때문임  
    -> 3은 Key의 개수(관측 시점 수)이지 성분 수가 아니므로 √3이 아님  
    -> 각 Key는 기온·습도 두 성분으로 되어 있어서 √1도 아님 (나누는 기준이 아님)  
  - **첫 행 최댓값** : 크기 조정 있음 0.337453, 크기 조정 없음 0.339158  

**Q2. Scaled Dot-Product Attention에서 크기 조정은 왜 하나요? 이번 예시에서 차이가 아주 크다고 말할 수 있나요?**

- 답변:  
  - **크기 조정을 하는 이유** : 내적은 여러 성분의 곱을 더하기 떄문에 성분 수가 커지면 점수의 규모도 커질 수 있다.   
    -> 점수 차이가 커지면 Softmax를 지나치게 한쪽으로 몰리게 할 수 있기 때문에 √d_k로 조정하여 완화함

**Q3. √d_k로 나누는 것만으로 가중치 합이 1이 되나요? 한 Query의 점수가 모두 같다면 Softmax 가중치는 어떻게 되나요?**

- 답변:  
  - **√d_k로 나누는 것만으로는 합이 1이 되지 않음**  
    -> √d_k로 나눈 것은 점수 규모의 조정을 의미  
    -> 양수 가중치로 만들고 행 합을 1로 만드는 과정은 **Softmax**임  
  - **점수가 모두 같다면 가중치도 모두 같아짐**  
    -> Key가 3개이면 각각 1 / 3 이 됨  
    -> 점수 자체를 단순히 그 합으로 나눈 방식은 softmax의 방식은 아니다.  

---

## 기초 과제. 다음 관측 구간에 Attention 적용하기


### 문제 3-1. 동일한 계산으로 다른 구간 요약하기

#### 문제 설명

`X_home`은 4번째부터 6번째 관측의 기온·상대습도를 같은 배율로 바꾼 입력입니다. WQ·WK·WV와 Softmax 함수를 그대로 사용하여 Self-Attention을 계산하세요.

이번 과제는 **새로운 3행 구간 안에서만** 계산합니다. 실습의 Q·K·V나 출력값을 이어 붙이지 않습니다. 가중치를 학습하거나 조정할 필요는 없습니다.

#### 요구사항

1. `Q_home`, `K_home`, `V_home`을 각각 만드세요.
2. `scores_home`, `scaled_home`, `weights_home`, `output_home`을 순서대로 계산하세요.
3. Q·K·V와 가중치·출력의 값과 크기를 출력하세요. 가중치의 행별 합도 확인하세요.
4. 첫 Query의 첫 내적 점수와 첫 가중합 출력을 수식에 대입해 확인하세요.
5. 첫 Query에서 가장 큰 비중의 관측 시각을 확인하고 출력의 의미를 설명하세요.

#### 작성 도움말

`d_home=K_home.shape[1]`을 사용합니다. `X_home`은 준비 셀에서 이미 배율을 조정했으므로 다시 나누지 않습니다. 첫 관측 시각은 00:40입니다.

#### 해석 질문

**Q1.** Q_home·K_home·V_home·weights_home·output_home의 크기는 각각 무엇인가요? 실습과 같은 크기가 되는 이유는 무엇인가요?

**Q2.** 첫 내적 점수와 첫 가중합 출력은 각각 얼마인가요? 대입 식을 쓰세요.

**Q3.** 첫 Query의 세 가중치와 가장 큰 비중의 시각은 무엇인가요? 전체 가중치의 각 행 합은 얼마인가요?

**Q4.** 결과 숫자가 실습과 달라진 것은 가중치를 새로 학습했기 때문인가요? 이 출력을 미래 기온 예측이라고 설명해도 되나요?

#### 제출 결과

단계별 계산·크기, 행별 가중치 합, 첫 행 대입 계산, Q1~Q4 답변. 모델 학습이나 심화 비교는 요구하지 않습니다.


In [ ]:
# 1. `Q_home`, `K_home`, `V_home`을 각각 만드세요.
# X_home에 같은 가중치 행렬을 적용하세요.
Q_home = X_home @ WQ
K_home = X_home @ WK
V_home = X_home @ WV

# 2. `scores_home`, `scaled_home`, `weights_home`, `output_home`을 순서대로 계산하세요.
# 내적, 크기 조정, Softmax, 가중합을 순서대로 계산하세요.

# 열 개수(d_k) 먼저 구하기 (스케일링: √(Key의 열 개수)로 나누기 때문)
d_home = K_home.shape[1]   # d_k : K_home의 열 수 = Key 한 줄의 숫자 개수 (shape[0]은 행 수, shape[1]은 열 수)

# 내적하기
scores_home = Q_home @ K_home.T
# Q_home @ K_home → (3, 2) @ (3, 2) : 가운데가 2와 3이라 계산 안 됨
# K_home.T : .T를 붙이면 행과 열을 뒤바꾼 배열로 바뀜
# → Q_home @ K_home.T → (3, 2) @ (2, 3) = (3, 3)으로 계산 될 수 있도록 만들기

# 스케일링
scaled_home = scores_home / np.sqrt(d_home)
# Q_home @ K_home 내적 / √(Key의 열 개수)

# softmax 함수 적용
weights_home = softmax_rows(scaled_home)

# V의 가중합 (Softmax로 구한 비중(weights_home)을 V_home에 곱해서 더한 것)
output_home = weights_home @ V_home

# 3. Q·K·V와 가중치·출력의 값과 크기를 출력하세요. 가중치의 행별 합도 확인하세요.
print("d_home:", d_home)
print("\nQ_home:\n", Q_home.round(6), "\nshape:", Q_home.shape)
print("\nK_home:\n", K_home.round(6), "\nshape:", K_home.shape)
print("\nV_home:\n", V_home.round(6), "\nshape:", V_home.shape)
print("\nscores_home:\n", scores_home.round(6), "\nshape:", scores_home.shape)
print("\nscaled_home:\n", scaled_home.round(6), "\nshape:", scaled_home.shape)
print("\nweights_home:\n", weights_home.round(6), "\nshape:", weights_home.shape)
print("\noutput_home:\n", output_home.round(6), "\nshape:", output_home.shape)
print("\n가중치 행별 합:", weights_home.sum(axis=1).round(6))
                                       # (axis=1) : Softmax가 각 행의 합을 1로 만들었는지 확인

# 4. 첫 Query의 첫 내적 점수와 첫 가중합 출력을 수식에 대입해 확인하세요.
s11 = Q_home[0, 0] * K_home[0, 0] + Q_home[0, 1] * K_home[0, 1]
print(f"\n[첫 내적] q1·k1 = ({Q_home[0,0]:.4f})×({K_home[0,0]:.4f}) + ({Q_home[0,1]:.4f})×({K_home[0,1]:.4f}) = {s11:.6f}")


contrib_home = weights_home[0] * V_home[:, 0]
print("[첫 가중합]")
for j in range(3):  # V_home의 시점 수 만큼 반복
    print(f"A[0,{j}] × V[{j},0] = {weights_home[0, j]:.6f} × ({V_home[j, 0]:.4f}) = {contrib_home[j]:.6f}")
print(f"o1 = {contrib_home.sum():.6f} / output_home[0,0] = {output_home[0, 0]:.6f}")


# 5. 첫 Query에서 가장 큰 비중의 관측 시각을 확인하고 출력의 의미를 설명하세요.
times_home = times.iloc[3:6].reset_index(drop=True)
max_j = weights_home[0].argmax()
print("\n첫 Query 가중치:", weights_home[0].round(6))
print("가장 큰 비중의 관측 시각:", times_home[max_j], "/ 비중:", round(weights_home[0, max_j], 6))

d_home: 2

Q_home:
 [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]] 
shape: (3, 2)

K_home:
 [[-0.831   0.471 ]
 [-0.827   0.4705]
 [-0.805   0.472 ]] 
shape: (3, 2)

V_home:
 [[-0.831]
 [-0.827]
 [-0.805]] 
shape: (3, 1)

scores_home:
 [[1.134243 1.130448 1.113579]
 [1.130448 1.126669 1.109887]
 [1.113579 1.109887 1.093593]] 
shape: (3, 3)

scaled_home:
 [[0.802031 0.799347 0.787419]
 [0.799347 0.796676 0.784809]
 [0.787419 0.784809 0.773287]] 
shape: (3, 3)

weights_home:
 [[0.335254 0.334355 0.330391]
 [0.335244 0.33435  0.330406]
 [0.335193 0.334319 0.330489]] 
shape: (3, 3)

output_home:
 [[-0.821072]
 [-0.821072]
 [-0.82107 ]] 
shape: (3, 1)

가중치 행별 합: [1. 1. 1.]

[첫 내적] q1·k1 = (-0.8310)×(-0.8310) + (0.9420)×(0.4710) = 1.134243
[첫 가중합]
A[0,0] × V[0,0] = 0.335254 × (-0.8310) = -0.278596
A[0,1] × V[1,0] = 0.334355 × (-0.8270) = -0.276512
A[0,2] × V[2,0] = 0.330391 × (-0.8050) = -0.265965
o1 = -0.821072 / output_home[0,0] = -0.821072

첫 Query 가중치: [0.335254 0.334355 0.330391]
가

#### 답변 작성란

**Q1. Q_home·K_home·V_home·weights_home·output_home의 크기는 각각 무엇인가요? 실습과 같은 크기가 되는 이유는 무엇인가요?**


- **크기** : Q_home (3, 2), K_home (3, 2), V_home (3, 1), weights_home (3, 3), output_home (3, 1)  

- **실습과 같은 크기인 이유** : 입력 X_home도 실습 X처럼 3개 시점 × 2개 성분(기온, 습도)인 (3, 2)이고, 같은 WQ, WK, WV를 썼기 때문임  
    -> 크기는 입력의 행, 열 수와 가중치 행렬의 크기로 정해지고, 값이 달라져도 크기는 바뀌지 않음  

**Q2. 첫 내적 점수와 첫 가중합 출력은 각각 얼마인가요? 대입 식을 쓰세요.**


- **첫 내적 점수 : 1.134243** (q1 = [-0.831, 0.942], k1 = [-0.831, 0.471])  
    -> (-0.831) × (-0.831) + 0.942 × 0.471 = 0.690561 + 0.443682 = 1.134243  
- **첫 가중합 출력 : -0.821072** (a1 = [0.335254, 0.334355, 0.330391], V_home = [-0.831, -0.827, -0.805])  
    -> o1 = 0.335254 × (-0.831) + 0.334355 × (-0.827) + 0.330391 × (-0.805)  
    -> = -0.278596 + (-0.276512) + (-0.265965) = 약 -0.821072 (반올림 차이)  

**Q3. 첫 Query의 세 가중치와 가장 큰 비중의 시각은 무엇인가요? 전체 가중치의 각 행 합은 얼마인가요?**


- **첫 Query 가중치 : [0.335254, 0.334355, 0.330391]**  
- **가장 큰 비중의 시각 : 00:40**  
    -> 첫 Query가 00:40 관측에 0.335254로 가장 큰 비중을 둠  
    -> 다만 세 비중이 0.330391 ~ 0.335254로 거의 비슷해서, 한 시점에 뚜렷하게 집중했다고 보기는 어려움  
- **각 행 합 : [1, 1, 1]**  
    -> Softmax가 각 행의 합이 1이 되도록 바꾸기 때문임  

**Q4. 결과 숫자가 실습과 달라진 것은 가중치를 새로 학습했기 때문인가요? 이 출력을 미래 기온 예측이라고 설명해도 되나요?**

- **새로 학습하지 않았음**  
    -> WQ, WK, WV와 Softmax 함수는 실습과 똑같고, 입력만 1~3번째 관측에서 4~6번째 관측(X_home)으로 바뀌었기 때문에 숫자가 달라짐  
    -> Attention 가중치(weights_home)가 달라진 것도 새로 학습했기 때문이 아니라, 바뀐 입력으로 Q와 K를 다시 계산했기 때문임  
- **미래 기온 예측이라고 설명할 수 없음**  
    -> 출력은 Attention으로 구한 가중치(weights_home)를 같은 구간 세 시점의 V(기온)에 곱해 더한 **가중평균**일 뿐, 다음 시점 기온을 예측하는 값이 아님  
    -> 예측하려면 출력층을 추가하고, 손실함수로 예측값과 실제 미래 기온의 차이를 계산해 이를 줄이도록 학습한 뒤 검증해 보아야 함

## 제출 전 확인

- 실습 2개와 기초 과제의 요구사항을 모두 수행했나요?
- 각 문제의 모든 해석 질문에 답했나요?
- Q·K·V의 역할과 Attention 가중치를 구분했나요?
- 내적·크기 조정·Softmax·가중합 순서를 지켰나요?
- 각 Query 행의 가중치 합이 1인지 확인했나요?
- 출력을 미래 기온 예측이나 인과관계로 해석하지 않았나요?


---

## 실습 마무리

실습 1·실습 2·기초 과제에서 배운 개념과 계산을 자신의 말로 정리하세요. 새로운 심화 문제가 아니라 **앞에서 사용한 수식과 결과를 다시 확인하는 복습**입니다. exp·제곱근의 값은 제공 코드나 앞의 계산 표를 참고해도 됩니다.

1. Attention과 Self-Attention은 각각 무엇인가요? 이번 계산이 Self-Attention인 이유와, Self가 자기 행만 참고한다는 뜻인지 설명하세요.  
- attention은 Q와 K를 비교하여 반영 비중을 정하고 그 비중으로 V를 가중합 하는 방식  
- self-attention은 QKV를 같은 입력 묶음에서 만드는 attention을 의미  
- 예제에서는 같은 기상 관측 세 행을 담은 X에 WQ, WK, WV를 각각 적용  
-> 출처가 같기 때문에 self attention


2. Q·K·V의 역할과 첫 행의 값, 전체 행렬 크기를 쓰세요. WQ·WK·WV와 Attention 가중치 A의 차이도 설명하세요.  
- Q : 비교의 기준, K : 그 기준과 비교할 표현, V : 비중대로 실제 합칠 정보를 의미  
- Q와 K는 (3,2), V는 (3,1), 첫 행은 Q1 = [-0.802, 0.933] K1 = [-0.802, 0.4655], V1 = [-0.802]  
- 첫 입력 [-0.802, 0.933]에서는 Q는 두 성분을 그대로 남긴다.  
- K의 둘째 성분은 0.933x0.5 = 0.4665  
- V는 -0.802x1 + 0.933x0 = -0.802  
- Q와 K는 같은 위치끼리 내적해야하기 때문에 성분 수가 같아야 하지만 V의 성분 수는 다를 수 있다.

3. 첫 Query와 첫 Key의 내적을 곱셈부터 계산하세요. Q@K.T에서 전치가 필요한 이유와 √2로 나누는 이유를 각각 설명하세요.  
- 첫 내적은 (-0.802)x(-0.802)+0.933x0.4665 = 0.643204 + 0.4352445 = 1.0784485  
- 원래 K는 3행 2열로 key마다 한 행을 의미, 행렬곱은 왼쪽 행과 오른쪽 열을 곱하기 때문에 K.T로 각 key를 열에 놓고  
- (3,2) @ (2,3)의 결과로 (3,3)이고 Q가 K와 비교된 점수는 9개  
- 루트 2로 나누는 이유 : d_k는 key의 하나의 성분 수이며 이번에는 2를 의미했기 떄문  

4. Softmax는 점수를 어떤 순서로 비중으로 바꾸나요? axis=1과 keepdims=True의 역할, 행별 합, 한 행의 점수가 모두 같을 때의 결과를 설명하세요.  
- 제공 함수는 행 별 최댓값을 빼고 -> 각 점수는 exp를 적용하고 -> 같은 행의 exp 값 합으로 나눈다.  
- axis = 1은 각 Q행 안에 세 참고 점수를 묶어 계산하라는 뜻  
- 항 행의 점수가 모두 같으면 최대값을 뺀 뒤 [0,0,0], exp 뒤 [1,1,1]이므로 결과는 각 1/3  
- 최대 점수에서 0이 나왔다고 비중이 0을 의미하지는 않음  

5. 첫 Query의 비중 [0.32755578, 0.33499149, 0.33745273]을 Value [−0.802, −0.841, −0.851]에 적용해 출력을 구하세요. 양수 비중에서 음수 출력이 나오는 이유와 출력 세 개의 의미도 설명하세요.  
- 0.3275578 x (-0.802) + 0.33499149 x (-0.841) + 0.33745273 x (-0.851)은  
약 -0.26269974 = 0.28717227 = -0.83159985  

6. 크기 조정 전후 첫 행의 최대 비중은 어떻게 달라지나요? 과제 구간에서 가장 큰 비중의 시각과 첫 출력을 쓰고, 이 결과가 학습·미래 예측·인과관계를 뜻하는지 설명하세요.  
- 첫 행의 최대 비중은 조정 전 약 0.33915807, 조정 후 약 0.33745273  
- 과제 첫 q에서 최대 비중은 0.40이며 첫 출력은 -0.8210722
